# Environment and RDN data check

Self-contained device, dataset integrity, class-count, and optional VMamba kernel checks. No classifier is trained and no dataset images are scored.

## 1. Imports

Use the same installed environment as the model notebooks.

In [ ]:
from __future__ import annotations

import contextlib
import copy
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
from PIL import Image
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm


## 2. Settings

The launcher sets RDN_DATA_ROOT. Change DATA_ROOT here when running independently.

In [ ]:
DATA_ROOT = Path(os.environ.get("RDN_DATA_ROOT", Path.home() / "Desktop" / "training_models"))
REQUIRE_CUDA = True
CHECK_VMAMBA_KERNEL = False


## 3. Device and data checks

Verify the exact raw data and label definition without training.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def atomic_checkpoint(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def environment_info(device):
    packages = {}
    for name in ("torch", "torchvision", "timm", "numpy", "scikit-learn",
                 "matplotlib", "Pillow", "safetensors", "mamba-ssm", "triton"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {"python": platform.python_version(), "platform": platform.platform(),
            "packages": packages, "cuda_runtime": torch.version.cuda,
            "device": str(device),
            "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
            "gpu_memory_bytes": torch.cuda.get_device_properties(device).total_memory
            if device.type == "cuda" else None}


def select_device(require_cuda=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        # Perform a tiny CUDA operation to catch wheels without this GPU's support.
        value = torch.ones(1, device=device) + 1
        if value.item() != 2:
            raise RuntimeError("CUDA check failed")
        return device
    if require_cuda:
        raise RuntimeError("CUDA is unavailable. Select the GPU environment/kernel; "
                           "use a CUDA-enabled Python environment. Training has not started.")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # VMamba custom kernels may not have deterministic implementations.
    torch.use_deterministic_algorithms(True, warn_only=True)


def precision_context(device, precision):
    if device.type != "cuda" or precision == "fp32":
        return contextlib.nullcontext()
    if precision == "bf16":
        if not torch.cuda.is_bf16_supported():
            raise RuntimeError("BF16 is unsupported on this GPU; set precision='fp32' or 'fp16'.")
        dtype = torch.bfloat16
    elif precision == "fp16":
        dtype = torch.float16
    else:
        raise ValueError("precision must be bf16, fp16, or fp32")
    return torch.autocast(device_type="cuda", dtype=dtype)

CLASS_NAMES = ["PeakBelow83", "PeakAtOrAbove83"]
EXPECTED_COUNTS = {"train": 97200, "validation": 10800, "test": 3000}
DATA_SHA256 = {'data/full_manifest.csv': '8b70f0ba4d746e9f504dd89ee03984602f725bfa9e3f359a1e82b46a9f2b0652',
 'data/images.npy': '4d9705401d6c2ca8c62da145506d5c2839c8a7285afd098b9c2fd575510df4da',
 'data/labels.npy': '8f58d927ca4199d703f6e8ada05597a86c50d86126c40ff94e878f7e58239638',
 'data/spectra.npy': '87168b2152ee172511a28da89648dec7366dece8e03310f91324a690819f62c9'}

def load_full_data(root):
    """Validate the exact packaged data, fixed target, and full split; no model evaluation."""
    root = Path(root)
    for name in ("data/images.npy", "data/labels.npy", "data/spectra.npy",
                 "data/full_manifest.csv"):
        actual = sha256_file(root / name)
        if actual != DATA_SHA256[name]:
            raise ValueError(f"File differs from the supplied full release: {name}")
    images = np.load(root / "data/images.npy", mmap_mode="r", allow_pickle=False)
    labels = np.load(root / "data/labels.npy", mmap_mode="r", allow_pickle=False)
    spectra = np.load(root / "data/spectra.npy", mmap_mode="r", allow_pickle=False)
    if images.shape != (111000, 16, 16) or labels.shape != (111000,) or spectra.shape != (111000, 128):
        raise ValueError("Expected the full 111,000-layout RDN release, not the pilot collection")
    if not np.isin(images, [0, 1]).all() or not np.isfinite(spectra).all():
        raise ValueError("Invalid image pixels or response values")
    if not np.array_equal(labels, (spectra.argmax(axis=1) >= 83).astype(np.int64)):
        raise ValueError("Labels must use the fixed compact-response cutoff at index 83")
    with (root / "data/full_manifest.csv").open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 111000 or [int(r["array_index"]) for r in rows] != list(range(111000)):
        raise ValueError("The full manifest must map every array index exactly once")
    if len({r["sample_id"] for r in rows}) != 111000:
        raise ValueError("Duplicate sample identifiers")
    if len({r["geometry_sha256"] for r in rows}) != 111000:
        raise ValueError("Duplicate geometries require a grouped split")
    partitions = {}
    for split, expected in EXPECTED_COUNTS.items():
        partition = [r for r in rows if r["split"] == split]
        if len(partition) != expected:
            raise ValueError(f"{split}: expected {expected} layouts, got {len(partition)}")
        indices = np.asarray([int(r["array_index"]) for r in partition], dtype=np.int64)
        expected_source = "test" if split == "test" else "train"
        if any(r["source_split"] != expected_source for r in partition):
            raise ValueError("Released test samples cannot enter training or validation")
        if not np.array_equal(labels[indices], np.asarray([int(r["label"]) for r in partition])):
            raise ValueError("Manifest labels disagree with label array")
        if split == "test" and not np.array_equal(indices, np.arange(108000, 111000)):
            raise ValueError("The original 3,000 released test samples must be preserved")
        if split != "test" and (indices >= 108000).any():
            raise ValueError("Test samples found in training/validation")
        partitions[split] = partition
    identity = {name: DATA_SHA256[name]
                for name in ("data/images.npy", "data/labels.npy", "data/spectra.npy",
                             "data/full_manifest.csv")}
    identity["target"] = "compact_response_argmax_ge_83_first_maximum"
    return partitions, identity


## 4. Optional VMamba kernel definitions

Only the optional kernel test uses these definitions. Set CHECK_VMAMBA_KERNEL=True before running the final cell to test CUDA forward/backward.

In [ ]:
from collections import OrderedDict
from functools import partial
from typing import Any, Callable, Optional
import warnings

import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from timm.layers import DropPath, trunc_normal_

try:
    import selective_scan_cuda
    WITH_SELECTIVESCAN_MAMBA = True
except ImportError:
    WITH_SELECTIVESCAN_MAMBA = False

def selective_scan_torch(
    u: torch.Tensor, # (B, K * C, L)
    delta: torch.Tensor, # (B, K * C, L)
    A: torch.Tensor, # (K * C, N)
    B: torch.Tensor, # (B, K, N, L)
    C: torch.Tensor, # (B, K, N, L)
    D: torch.Tensor = None, # (K * C)
    delta_bias: torch.Tensor = None, # (K * C)
    delta_softplus=True, 
    oflex=True, 
    *args,
    **kwargs
):
    dtype_in = u.dtype
    Batch, K, N, L = B.shape
    KCdim = u.shape[1]
    Cdim = int(KCdim / K)
    assert u.shape == (Batch, KCdim, L)
    assert delta.shape == (Batch, KCdim, L)
    assert A.shape == (KCdim, N)
    assert C.shape == B.shape

    if delta_bias is not None:
        delta = delta + delta_bias[..., None]
    if delta_softplus:
        delta = torch.nn.functional.softplus(delta)

    u, delta, A, B, C = u.float(), delta.float(), A.float(), B.float(), C.float()
    B = B.view(Batch, K, 1, N, L).repeat(1, 1, Cdim, 1, 1).view(Batch, KCdim, N, L)
    C = C.view(Batch, K, 1, N, L).repeat(1, 1, Cdim, 1, 1).view(Batch, KCdim, N, L)
    deltaA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A))
    deltaB_u = torch.einsum('bdl,bdnl,bdl->bdln', delta, B, u)

    if True:
        x = A.new_zeros((Batch, KCdim, N))
        ys = []
        for i in range(L):
            x = deltaA[:, :, i, :] * x + deltaB_u[:, :, i, :]
            y = torch.einsum('bdn,bdn->bd', x, C[:, :, :, i])
            ys.append(y)
        y = torch.stack(ys, dim=2) # (B, C, L)

    out = y if D is None else y + u * D.unsqueeze(-1)
    return out if oflex else out.to(dtype=dtype_in)

class SelectiveScanCuda(torch.autograd.Function):
    @staticmethod
    @torch.amp.custom_fwd(device_type="cuda")
    def forward(ctx, u, delta, A, B, C, D=None, delta_bias=None,
                delta_softplus=False, oflex=True, backend=None):
        if backend not in (None, "mamba"):
            raise ValueError("This notebook uses the Mamba selective-scan extension.")
        ctx.delta_softplus = delta_softplus
        out, state, *_ = selective_scan_cuda.fwd(
            u, delta, A, B, C, D, None, delta_bias, delta_softplus,
        )
        ctx.save_for_backward(u, delta, A, B, C, D, delta_bias, state)
        return out

    @staticmethod
    @torch.amp.custom_bwd(device_type="cuda")
    def backward(ctx, dout):
        u, delta, A, B, C, D, delta_bias, state = ctx.saved_tensors
        if dout.stride(-1) != 1:
            dout = dout.contiguous()
        du, ddelta, dA, dB, dC, dD, dbias, *_ = selective_scan_cuda.bwd(
            u, delta, A, B, C, D, None, delta_bias, dout, state,
            None, None, ctx.delta_softplus, False,
        )
        return du, ddelta, dA, dB, dC, dD, dbias, None, None, None

def selective_scan_fn(
    u: torch.Tensor, # (B, K * C, L)
    delta: torch.Tensor, # (B, K * C, L)
    A: torch.Tensor, # (K * C, N)
    B: torch.Tensor, # (B, K, N, L)
    C: torch.Tensor, # (B, K, N, L)
    D: torch.Tensor = None, # (K * C)
    delta_bias: torch.Tensor = None, # (K * C)
    delta_softplus=True, 
    oflex=True,
    backend=None,
):
    fn = selective_scan_torch if backend == "torch" or (not WITH_SELECTIVESCAN_MAMBA) else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)

def check_scan():
    if not torch.cuda.is_available() or not WITH_SELECTIVESCAN_MAMBA:
        raise RuntimeError('Install the supported CUDA selective_scan_cuda extension first; see README')
    for dtype in (torch.float32, torch.bfloat16):
        torch.manual_seed(42)
        values = [torch.randn((1, 8, 16), device='cuda', dtype=dtype) * .1,
                  torch.randn((1, 8, 16), device='cuda', dtype=dtype) * .1,
                  -torch.rand((8, 1), device='cuda', dtype=torch.float32),
                  torch.randn((1, 4, 1, 16), device='cuda', dtype=dtype) * .1,
                  torch.randn((1, 4, 1, 16), device='cuda', dtype=dtype) * .1,
                  torch.randn(8, device='cuda', dtype=torch.float32) * .1,
                  torch.randn(8, device='cuda', dtype=torch.float32) * .1]
        ref_values = [x.detach().clone().requires_grad_() for x in values]
        actual_values = [x.detach().clone().requires_grad_() for x in values]
        reference = selective_scan_fn(*ref_values, delta_softplus=True, oflex=True, backend='torch')
        actual = selective_scan_fn(*actual_values, delta_softplus=True, oflex=True, backend='mamba')
        tol = .02 if dtype == torch.bfloat16 else 1e-4
        torch.testing.assert_close(actual.float(), reference.float(), rtol=tol, atol=tol)
        reference.float().square().mean().backward(); actual.float().square().mean().backward()
        for ref, out in zip(ref_values, actual_values):
            assert out.grad is not None and torch.isfinite(out.grad).all()
            torch.testing.assert_close(out.grad.float(), ref.grad.float(), rtol=max(tol, .02), atol=max(tol, 1e-4))


## 5. Run checks

Report both class counts in every split. These are integrity checks, not model accuracy.

In [ ]:
device = select_device(require_cuda=REQUIRE_CUDA)
partitions, identity = load_full_data(DATA_ROOT)
if CHECK_VMAMBA_KERNEL:
    check_scan()
report = {
    "environment": environment_info(device),
    "split": {name: len(rows) for name, rows in partitions.items()},
    "class_counts": {name: np.bincount([int(row["label"]) for row in rows], minlength=2).tolist()
                     for name, rows in partitions.items()},
    "vmamba_kernel_checked": CHECK_VMAMBA_KERNEL,
    "training_executed": False,
}
print(json.dumps(report, indent=2))
